### Import

In [1]:
import os
import warnings
from pathlib import Path
from dotenv import load_dotenv
import logging
import sys
import numpy as np
import pandas as pd
from numpy.typing import NDArray
from collections.abc import Sequence
from typing import Callable
from sklearn.preprocessing import MinMaxScaler
from dataclasses import dataclass

### Config Settings

In [2]:
# Resolve project root (this notebook lives in notebooks/) and load .env
# from the root, mirroring src/msdl_jci/config/settings.py behaviour.
_NB = Path.cwd().resolve()
while not (_NB / "pyproject.toml").exists() and _NB != _NB.parent:
    _NB = _NB.parent
load_dotenv(_NB / ".env")


class Config:
    """Minimal settings mirror for this dev notebook.

    Attribute names match src/msdl_jci/config/settings.py so the
    MultiSourceDatasetBuilder below runs unchanged. Values carry the
    same working defaults (lookback 28, horizon 5, 768-d news).
    """

    _raw_db_path = os.getenv("DATABASE_PATH", "./database/main_database.db")
    if Path(_raw_db_path).is_absolute():
        DB_FULL_PATH = Path(_raw_db_path)
    else:
        DB_FULL_PATH = _NB / _raw_db_path

    LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO").upper()

    # Data locations (defaults match the canonical pipeline)
    JCI_HISTORICAL_CSV = _NB / "data" / "raw" / "jci_historical.csv"
    BI_RATE_CSV = _NB / "data" / "raw" / "bi_rate.csv"
    INFLATION_CSV = _NB / "data" / "raw" / "inflation_data.csv"
    KURS_CSV = _NB / "data" / "raw" / "kurs_usdidr.csv"
    DAILY_NEWS_EMBEDDINGS_CSV = (
        _NB / "data" / "processed" / "daily_news_embeddings.csv"
    )

    # Model hyperparameters (Thesis Table 5 defaults)
    ML_LOOK_BACK: int = int(os.getenv("ML_LOOK_BACK", "28"))
    ML_PREDICTION_HORIZON: int = int(os.getenv("ML_PREDICTION_HORIZON", "5"))
    NEWS_EMB_DIM: int = int(os.getenv("NEWS_EMB_DIM", "768"))

    @classmethod
    def get_db_url(cls) -> str:
        return f"sqlite:///{cls.DB_FULL_PATH}"

    @classmethod
    def validate_db_exists(cls):
        if not cls.DB_FULL_PATH.exists():
            raise FileNotFoundError(
                f"Database file not found at {cls.DB_FULL_PATH}\n"
                f"Check your '.env' file or ensure the './database/' folder exists."
            )

### Constant

In [3]:
_LOG_FMT = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"

_DATE_FMT = "%Y-%m-%d %H:%M:%S"

MACRO_BASE_COLS = ["bi_rate", "inflation_rate", "usd_idr"]

CHANGE_LAGS = (21, 63, 126)

ZSCORE_WINDOWS = (126, 252)

EPSILON = 1e-9

MONTH_MAP = {
    "januari": "01",
    "jan": "01",
    "februari": "02",
    "feb": "02",
    "maret": "03",
    "mar": "03",
    "april": "04",
    "apr": "04",
    "mei": "05",
    "may": "05",
    "juni": "06",
    "jun": "06",
    "juli": "07",
    "jul": "07",
    "agustus": "08",
    "agu": "08",
    "aug": "08",
    "september": "09",
    "sep": "09",
    "oktober": "10",
    "okt": "10",
    "oct": "10",
    "november": "11",
    "nov": "11",
    "desember": "12",
    "des": "12",
    "dec": "12",
}

# Machine Learning / Modeling Hyperparameters (Aligned with Thesis Table 5)
ML_LOOK_BACK: int = int(os.getenv("ML_LOOK_BACK", "28"))
ML_PREDICTION_HORIZON: int = int(os.getenv("ML_PREDICTION_HORIZON", "5"))
ML_HIDDEN_LAYERS: tuple[int, ...] = (50, 25)
ML_MAX_ITER: int = int(os.getenv("ML_MAX_ITER", "500"))
ML_RANDOM_STATE: int = int(os.getenv("ML_RANDOM_STATE", "42"))

print("ML LOOK BACK:", ML_LOOK_BACK)
print("ML PREDICTION HORIZON:", ML_PREDICTION_HORIZON)
print("ML HIDDEN LAYERS:", ML_HIDDEN_LAYERS)
print("ML MAX ITER:", ML_MAX_ITER)
print("ML RANDOM STATE:", ML_RANDOM_STATE)

ML LOOK BACK: 28
ML PREDICTION HORIZON: 5
ML HIDDEN LAYERS: (50, 25)
ML MAX ITER: 500
ML RANDOM STATE: 42


### Utility

#### Data Loader

In [4]:
def read_numeric_series(
    csv_path: str | Path,
    column_name: str,
    remove_symbol: str = "",
) -> NDArray[np.float64]:
    path_obj = Path(csv_path)
    if not path_obj.exists():
        raise FileNotFoundError(f"File not found at path: {path_obj}")

    df = pd.read_csv(path_obj)

    if column_name not in df.columns:
        raise KeyError(f"Column '{column_name}' not found. Available: {list(df.columns)}")

    series = df[column_name].astype(str).str.strip()
    if remove_symbol:
        series = series.str.replace(remove_symbol, "", regex=False).str.replace(
            ",", "", regex=False
        )

    return pd.to_numeric(series, errors="raise").to_numpy(dtype=np.float64)

#### Logging Config

In [5]:
def configure_logging(level: int | str = logging.INFO) -> None:
    root = logging.getLogger()
    if isinstance(level, str):
        level = getattr(logging, level.upper(), logging.INFO)
    root.setLevel(level)

    for handler in root.handlers[:]:
        root.removeHandler(handler)

    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(level)
    handler.setFormatter(logging.Formatter(_LOG_FMT, datefmt=_DATE_FMT))
    root.addHandler(handler)


def get_logger(name: str) -> logging.Logger:
    return logging.getLogger(name)

In [6]:
def _safe_pct_change(series: pd.Series, lag: int) -> pd.Series:
    prev = series.shift(lag)
    return (series - prev) / (prev.abs() + EPSILON)


def _safe_zscore(series: pd.Series, window: int) -> pd.Series:
    min_periods = max(21, window // 4)
    roll = series.rolling(window, min_periods=min_periods)
    mu = roll.mean()
    sd = roll.std().replace(0.0, np.nan)
    return ((series - mu) / (sd + EPSILON)).fillna(0.0)


def add_macro_change_features(
    df: pd.DataFrame,
    lags: Sequence[int] | None = None,
    z_windows: Sequence[int] | None = None,
) -> pd.DataFrame:
    if lags is None:
        lags = CHANGE_LAGS
    if z_windows is None:
        z_windows = ZSCORE_WINDOWS

    result = df.copy().sort_values("Date").reset_index(drop=True)
    shifted = {col: result[col].shift(1) for col in MACRO_BASE_COLS if col in result.columns}

    feature_map: dict[str, Callable[[pd.Series], pd.Series]] = {}
    for col, series in shifted.items():
        for lag in lags:
            key = f"{col}_chg_{lag}d"
            feature_map[key] = lambda s=series, l=lag: _safe_pct_change(s, l)
        for w in z_windows:
            key = f"{col}_z{w}d"
            feature_map[key] = lambda s=series, w=w: _safe_zscore(s, w)

    for name, func in feature_map.items():
        if name not in result.columns:
            result[name] = func()

    if "bi_rate_chg_63d" in result.columns:
        result["rate_hiking"] = (result["bi_rate_chg_63d"] > 0.02).astype(float)
        result["rate_cutting"] = (result["bi_rate_chg_63d"] < -0.02).astype(float)
    if "inflation_rate" in result.columns:
        result["high_inflation"] = (result["inflation_rate"].shift(1) > 5.0).astype(float)
    if "usd_idr_chg_63d" in result.columns:
        result["idr_depreciating"] = (result["usd_idr_chg_63d"] > 0.02).astype(float)

    return result


ENGINEERED_MACRO_COLS = [
    *[f"bi_rate_chg_{l}d" for l in CHANGE_LAGS],
    *[f"inflation_rate_chg_{l}d" for l in CHANGE_LAGS],
    *[f"usd_idr_chg_{l}d" for l in CHANGE_LAGS],
    *[f"usd_idr_z{w}d" for w in ZSCORE_WINDOWS],
    "rate_hiking",
    "rate_cutting",
    "high_inflation",
    "idr_depreciating",
]

In [7]:
logger = get_logger(__name__)


@dataclass(slots=True)
class MultiSourceTensors:
    X_tech: np.ndarray
    X_macro: np.ndarray
    X_news: np.ndarray
    y: np.ndarray
    returns_5d: np.ndarray
    dates: np.ndarray
    close_prices: np.ndarray
    tech_feature_cols: list[str]


def _parse_id_date(s: pd.Series, day_first: bool = True) -> pd.Series:
    s = s.astype(str).str.strip().str.lower()
    for k, v in MONTH_MAP.items():
        s = s.str.replace(k, v, regex=False)
    return pd.to_datetime(s, errors="coerce", dayfirst=day_first)


def _calc_tech_indicators(df: pd.DataFrame) -> pd.DataFrame:
    c = df["Close"].astype(float)
    h, l, v = df["High"].astype(float), df["Low"].astype(float), df["Volume"].astype(float)

    # RSI-14
    d = c.diff()
    g, lo = d.clip(lower=0), -d.clip(upper=0)
    ag = g.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
    al = lo.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
    df["RSI_14"] = 100 - 100 / (1 + ag / (al + 1e-9))

    # MACD (12,26,9)
    e12 = c.ewm(span=12, adjust=False).mean()
    e26 = c.ewm(span=26, adjust=False).mean()
    df["MACD"] = e12 - e26
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

    # ATR-14
    pc = c.shift(1)
    tr = pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)
    df["ATR_14"] = tr.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()

    # SMA-20
    df["SMA_20"] = c.rolling(20).mean()
    return df


class MultiSourceDatasetBuilder:
    TECH_COLS = ["Close", "Volume", "RSI_14", "MACD", "MACD_Signal", "ATR_14", "SMA_20"]
    MACRO_COLS = ["bi_rate", "inflation_rate", "usd_idr"]

    def __init__(self, look_back: int | None = None, horizon: int | None = None):
        cfg = Config()
        self.look_back = look_back or cfg.ML_LOOK_BACK
        self.horizon = horizon or cfg.ML_PREDICTION_HORIZON

    def _load_macro(self, bi_p, inf_p, kurs_p):
        bi = pd.read_csv(bi_p)
        bc = "BI-7Day-RR" if "BI-7Day-RR" in bi.columns else bi.columns[-1]
        bp = "Period" if "Period" in bi.columns else bi.columns[1]
        bi["Date"] = _parse_id_date(bi[bp])
        bi["bi_rate"] = (
            bi[bc].astype(str).str.replace("%", "", regex=False).str.strip().astype(float)
        )

        inf = pd.read_csv(inf_p)
        ic = "Data Inflasi" if "Data Inflasi" in inf.columns else inf.columns[-1]
        ip = "Periode" if "Periode" in inf.columns else inf.columns[0]
        inf["Date"] = _parse_id_date(inf[ip], day_first=False)
        inf["inflation_rate"] = (
            inf[ic].astype(str).str.replace("%", "", regex=False).str.strip().astype(float)
        )

        kur = pd.read_csv(kurs_p)
        kur["Date"] = pd.to_datetime(kur["Date"], errors="coerce")
        kur["usd_idr"] = pd.to_numeric(
            kur["Close"].astype(str).str.replace(",", "", regex=False), errors="coerce"
        )

        return (
            bi.dropna(subset=["Date"]).sort_values("Date")[["Date", "bi_rate"]],
            inf.dropna(subset=["Date"]).sort_values("Date")[["Date", "inflation_rate"]],
            kur.dropna(subset=["Date"]).sort_values("Date")[["Date", "usd_idr"]],
        )

    def build_aligned_dataframe(
        self, jci_csv=None, bi_csv=None, inf_csv=None, kurs_csv=None, news_csv=None
    ):
        cfg = Config()
        jci = pd.read_csv(jci_csv or cfg.JCI_HISTORICAL_CSV)
        jci["Date"] = pd.to_datetime(jci["Date"], errors="coerce")
        jci = jci.dropna(subset=["Date", "Close"]).sort_values("Date").reset_index(drop=True)
        _calc_tech_indicators(jci)

        bi, inf, kur = self._load_macro(
            bi_csv or cfg.BI_RATE_CSV,
            inf_csv or cfg.INFLATION_CSV,
            kurs_csv or cfg.KURS_CSV,
        )

        m = pd.merge(jci, kur, on="Date", how="left")
        m = pd.merge_asof(
            m.sort_values("Date"), bi.sort_values("Date"), on="Date", direction="backward"
        )
        m = pd.merge_asof(
            m.sort_values("Date"), inf.sort_values("Date"), on="Date", direction="backward"
        )

        for col in self.MACRO_COLS:
            m[col] = m[col].ffill()
        m = m.dropna(subset=self.MACRO_COLS).reset_index(drop=True)

        fc = m["Close"].shift(-self.horizon)
        m["return_5d"] = (fc - m["Close"]) / m["Close"]
        m["target_direction"] = (fc > m["Close"]).astype(float)
        m["future_close"] = fc

        news_p = news_csv or cfg.DAILY_NEWS_EMBEDDINGS_CSV
        emb_cols = [c for c in m.columns if c.startswith("emb_")]
        if not emb_cols:
            if Path(news_p).exists():
                ndf = pd.read_csv(news_p).rename(columns={"trade_date": "Date"})
                ndf["Date"] = pd.to_datetime(ndf["Date"], format="%Y-%m-%d", errors="coerce")
                ec = [c for c in ndf.columns if c.startswith("emb_")]
                m = pd.merge(m, ndf, on="Date", how="left")
                m[ec] = m[ec].fillna(0.0)
            else:
                ec = [f"emb_{i}" for i in range(cfg.NEWS_EMB_DIM)]
                m[ec] = 0.0
            emb_cols = ec
        else:
            m[emb_cols] = m[emb_cols].fillna(0.0)

        m = m.dropna(subset=self.TECH_COLS + ["future_close"]).reset_index(drop=True)
        logger.info("Aligned: %d samples (%s → %s)", len(m), m["Date"].min(), m["Date"].max())
        return m

    def create_multisource_tensors(
        self, df, feat_scaler=None, macro_scaler=None, fit_scalers=True, train_end_idx=None
    ):
        # Leakage guard: future_close must never become a feature.
        forbidden = {"future_close", "return_5d", "target_direction"}
        leaked = forbidden.intersection(set(self.TECH_COLS))
        if leaked:
            raise ValueError(f"TECH_COLS must not contain target columns: {leaked}")

        emb_cols = [c for c in df.columns if c.startswith("emb_")]
        raw_tech = df[self.TECH_COLS].values.astype(np.float32)
        raw_macro = df[self.MACRO_COLS].values.astype(np.float32)
        raw_news = df[emb_cols].values.astype(np.float32)
        lb = self.look_back
        n = len(df) - lb + 1
        if n <= 0:
            raise ValueError(f"Need ≥{lb} rows, got {len(df)}")

        if fit_scalers:
            feat_scaler = MinMaxScaler(feature_range=(0, 1))
            macro_scaler = MinMaxScaler(feature_range=(0, 1))
            if train_end_idx is not None:
                if not (0 < train_end_idx <= len(df)):
                    raise ValueError(f"train_end_idx={train_end_idx} out of range")
                end = train_end_idx
                feat_scaler.fit(raw_tech[:end])
                macro_scaler.fit(raw_macro[:end])
                logger.info("Scalers fit on [: %d] / %d rows", end, len(df))
            else:
                warnings.warn(
                    "Fitting scalers on FULL dataframe — leaks test ranges. "
                    "Pass train_end_idx for leak-free evaluation."
                )
                feat_scaler.fit(raw_tech)
                macro_scaler.fit(raw_macro)
                logger.info("Scalers fit on full frame (%d rows)", len(df))
        elif feat_scaler is None or macro_scaler is None:
            raise ValueError("Provide fitted scalers when fit_scalers=False")

        st = feat_scaler.transform(raw_tech)
        sm = macro_scaler.transform(raw_macro)

        # Sliding window construction: [Samples, Lookback, D_tech]
        X_tech_list, X_macro_list, X_news_list = [], [], []
        y_list, returns_list, dates_list, close_list = [], [], [], []
        for i in range(lb - 1, len(df)):
            X_tech_list.append(st[i - lb + 1 : i + 1])
            X_macro_list.append(sm[i])
            X_news_list.append(raw_news[i])
            y_list.append(df["target_direction"].values[i])
            returns_list.append(df["return_5d"].values[i])
            dates_list.append(df["Date"].dt.strftime("%Y-%m-%d").values[i])
            close_list.append(df["Close"].values[i])
        tensors = MultiSourceTensors(
            X_tech=np.array(X_tech_list, dtype=np.float32),
            X_macro=np.array(X_macro_list, dtype=np.float32),
            X_news=np.array(X_news_list, dtype=np.float32),
            y=np.array(y_list, dtype=np.float32),
            returns_5d=np.array(returns_list, dtype=np.float32),
            dates=np.array(dates_list),
            close_prices=np.array(close_list, dtype=np.float32),
            tech_feature_cols=self.TECH_COLS,
        )
        return tensors, feat_scaler, macro_scaler

### Smoke demo (synthetic, no raw CSVs needed)

Exercises the builder end-to-end on synthetic data: train-only scaler fit,
leakage guard, and windowed tensor shapes. Methodology mirrors AGENTS.md —
ffill-only macro alignment, t+5 labels, `train_end_idx` scoping, seed 42.

In [8]:
np.random.seed(42)
n_days = 120
dates = pd.date_range("2024-01-01", periods=n_days, freq="B")
close = 7000 + np.cumsum(np.random.randn(n_days) * 20)
demo = pd.DataFrame({
    "Date": dates,
    "Close": close,
    "High": close + np.abs(np.random.randn(n_days) * 10),
    "Low": close - np.abs(np.random.randn(n_days) * 10),
    "Volume": np.abs(np.random.randn(n_days) * 1e6) + 1e6,
    "bi_rate": 6.0,
    "inflation_rate": 2.5,
    "usd_idr": 15600.0,
})
_calc_tech_indicators(demo)
H = ML_PREDICTION_HORIZON
fc = demo["Close"].shift(-H)
demo["return_5d"] = (fc - demo["Close"]) / demo["Close"]
demo["target_direction"] = (fc > demo["Close"]).astype(float)
demo["future_close"] = fc
for i in range(8):
    demo[f"emb_{i}"] = 0.0
demo = demo.dropna(subset=MultiSourceDatasetBuilder.TECH_COLS + ["future_close"]).reset_index(drop=True)

builder = MultiSourceDatasetBuilder()
lb = builder.look_back
n_win = len(demo) - lb + 1
train_end_win = int(n_win * 0.70)
train_end_df = train_end_win + lb - 1  # window space -> df space
tensors, _, _ = builder.create_multisource_tensors(demo, train_end_idx=train_end_df)
print("samples:", len(tensors.y), "| X_tech:", tensors.X_tech.shape)
print("X_macro:", tensors.X_macro.shape, "| X_news:", tensors.X_news.shape)
print("pos_rate:", round(float(tensors.y.mean()), 3))
assert tensors.X_tech.shape[1:] == (lb, len(builder.TECH_COLS))
assert len(tensors.y) == n_win
assert np.isfinite(tensors.X_tech).all(), "NaN/inf leaked into X_tech"
assert np.isfinite(tensors.X_macro).all(), "NaN/inf leaked into X_macro"
print("OK: leak-free smoke demo passed")


samples: 69 | X_tech: (69, 28, 7)
X_macro: (69, 3) | X_news: (69, 8)
pos_rate: 0.522
OK: leak-free smoke demo passed
